In [ ]:
"""
TravelBuddy Chatbot
-------------------
A travel assistant using:
- JSON-based intents
- Regex pattern matching
- Memory system
- Full country info display
- Named regex groups
"""

import json
import re
import random

# Load intents and travel data 
def load_intents(filename="intents.json"):
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

data = load_intents()
intents = data["intents"]
travel_data = data.get("travel_data", {})

# Memory
memory = {}

#Compile regex
for intent in intents:
    intent["_regex"] = [re.compile(p, re.IGNORECASE) for p in intent.get("patterns", [])]

# Match intent
def match_intent(user_input):
    for intent in intents:
        for pattern in intent["_regex"]:
            match = pattern.match(user_input)
            if match:
                return intent, match.groupdict()
    return None, {}

#  Generate responsee
def respond(user_input):
    intent, groups = match_intent(user_input)

    if not intent:
        intent = next(i for i in intents if i["tag"] == "fallback")
        groups = {}

    # Store user's name
    if intent["tag"] == "remember_name" and "name" in groups:
        memory["name"] = groups["name"]

    # Recall user's name
    if intent["tag"] == "recall_name":
        if "name" in memory:
            return random.choice(intent["responses"]).replace("{name}", memory["name"])
        else:
            return "I don't know your name yet. What should I call you?"
    # Feeling intent with named group
    if intent["tag"] == "feeling":
        response = random.choice(intent["responses"])
        # Replace {feeling} if the group exists
        if "feeling" in groups:
            response = response.replace("{feeling}", groups["feeling"])
        else:
            # If no feeling captured, remove placeholder
            response = response.replace("{feeling}", "good")
        return response


    # Continent, list of countries
    if intent["tag"] == "continent_interest" and "continent" in groups:
        continent = groups["continent"].title()
        countries = [c for c, info in travel_data.items() if info["continent"].lower() == continent.lower()]
        if countries:
            return f"{continent} has these countries: {', '.join(countries)}. Which one interests you?"
        else:
            return f"Sorry, I don't have data for {continent} yet."

    # Country, full info
    if intent["tag"] == "country_interest" and "country" in groups:
        country = groups["country"].title()
        info = travel_data.get(country)
        if info:
            attractions = ", ".join(info.get("things_to_do", []))
            cities = ", ".join(info.get("cities", []))
            visa = ", ".join(f"{k}: {v}" for k, v in info.get("visa_info", {}).items())
            difficulty = info.get("difficulty", "N/A")
            budget = ", ".join(f"{k}: {v}" for k, v in info.get("budget", {}).items())
            flights = ", ".join(info.get("flights", []))
            return (
                f"{country} is in {info['continent']}.\n"
                f"Cities: {cities}\n"
                f"Things to do: {attractions}\n"
                f"Visa info: {visa}\n"
                f"Difficulty: {difficulty}\n"
                f"Budget: {budget}\n"
                f"Flights: {flights}"
            )
        else:
            return (f"I don’t have detailed information about {country} yet, "
                    "but I can still help you plan! What would you like to know — "
                    "flights, weather, budget, or things to do?")




    # Default response
    response = random.choice(intent["responses"])
    for key, value in groups.items():
        response = response.replace(f"{{{key}}}", value)
    if "{name}" in response and "name" in memory:
        response = response.replace("{name}", memory["name"])
    return response

# Main Loop

def main():
    print("TravelBuddy: Hello! I'm TravelBuddy — your travel assistant.")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("TravelBuddy: Goodbye! Safe travels!")
            break
        reply = respond(user_input)
        print(f"TravelBuddy: {reply}\n")

if __name__ == "__main__":
    main()


TravelBuddy: Hello! I'm TravelBuddy — your travel assistant.


You:  Hi


TravelBuddy: Hello traveler! Where would you like to go next?



You:  I want to go to Sri Lanka


TravelBuddy: I don’t have detailed information about Sri Lanka yet, but I can still help you plan! What would you like to know — flights, weather, budget, or things to do?

